# Preprocessing

Structural cleaning of `tovima.csv` per the audit findings.
Output: `tovima_clean.csv`. No NLP-level cleaning here.

In [1]:
import pandas as pd
import numpy as np
import re

pd.set_option("display.max_colwidth", 80)

## 0. Load raw data

In [2]:
df = pd.read_csv("tovima.csv", encoding="utf-32", sep="\t")
print("Raw shape:", df.shape)
n_start = len(df)

Raw shape: (14277, 5)


## 1. Strip leading-space scraper artifact

In [3]:
for col in df.columns:
    df[col] = df[col].astype(str).str.strip()

## 2. Drop unparseable dates (audit #2)

In [4]:
df["Date_parsed"] = pd.to_datetime(df["Date"], errors="coerce", utc=True)
n_bad_dates = df["Date_parsed"].isna().sum()
print(f"Dropping {n_bad_dates} rows with unparseable dates")

df = df[df["Date_parsed"].notna()].reset_index(drop=True)
print("Shape after dropping bad dates:", df.shape)

Dropping 3 rows with unparseable dates
Shape after dropping bad dates: (14274, 6)


## 3. Drop 2017 partial year (audit #3); keep 2023

In [5]:
df["Year"] = df["Date_parsed"].dt.year
n_2017 = (df["Year"] == 2017).sum()
print(f"Dropping {n_2017} rows from 2017 (partial year, scraper start-up)")

df = df[df["Year"] != 2017].reset_index(drop=True)
print("Shape after dropping 2017:", df.shape)
print()
print("Remaining year coverage:")
print(df["Year"].value_counts().sort_index())
print()
print("NOTE for later analysis: 2023 covers Jan\u2013Jul only (7 of 12 months).")
print("Treat its raw count as ~7/12 of a comparable full year when reading")
print("any year-over-year frequency chart.")

Dropping 10 rows from 2017 (partial year, scraper start-up)
Shape after dropping 2017: (14264, 7)

Remaining year coverage:
Year
2018    2360
2019    2721
2020    2490
2021    2518
2022    2706
2023    1469
Name: count, dtype: int64

NOTE for later analysis: 2023 covers Jan–Jul only (7 of 12 months).
Treat its raw count as ~7/12 of a comparable full year when reading
any year-over-year frequency chart.


## 4. Deduplicate on (Subject, Message) content (audit #1)

Keep earliest occurrence.

In [6]:
n_before = len(df)
df = df.sort_values("Date_parsed").drop_duplicates(subset=["Subject", "Message"], keep="first")
df = df.reset_index(drop=True)
n_after = len(df)
print(f"Dropped {n_before - n_after} content-duplicate rows ({n_before} -> {n_after})")

Dropped 326 content-duplicate rows (14264 -> 13938)


## 5. Strip HTML/CSS remnants (audit #5)

In [7]:
HTML_REGEX = re.compile(r"<[^>]+>")
STYLE_REGEX = re.compile(r"(style=|width:|padding:|border:)", re.IGNORECASE)

has_html = df["Message"].fillna("").str.contains(HTML_REGEX, regex=True)
has_style = df["Message"].fillna("").str.contains(STYLE_REGEX, regex=True)
df["had_html_markup"] = has_html | has_style

print(f"Rows with HTML/CSS markup: {df['had_html_markup'].sum()} ({df['had_html_markup'].mean()*100:.1f}%)")


def strip_html(text):
    if not isinstance(text, str):
        return text
    text = HTML_REGEX.sub(" ", text)
    return text


df["Message"] = df["Message"].apply(strip_html)

C:\Users\Jim\AppData\Local\Temp\ipykernel_13700\754633777.py:5: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  has_style = df["Message"].fillna("").str.contains(STYLE_REGEX, regex=True)


Rows with HTML/CSS markup: 2853 (20.5%)


Tags removed here; residual CSS-looking tokens handled later at token level. `had_html_markup` flag kept.

## 6. Flag (don't strip) reply/forward chains (audit #6)

In [8]:
reply_pattern = re.compile(r"\bRE:|\bΑΠ:|\bFW:|\bΠΡΘ:", re.IGNORECASE)
df["is_reply_or_forward"] = df["Subject"].str.contains(reply_pattern, regex=True)
print(f"Reply/forward subjects: {df['is_reply_or_forward'].sum()} ({df['is_reply_or_forward'].mean()*100:.1f}%)")

Reply/forward subjects: 887 (6.4%)


## 7. Drop near-empty messages (<20 chars)

In [9]:
msg_len = df["Message"].fillna("").str.len()
n_short = (msg_len < 20).sum()
print(f"Dropping {n_short} rows with message body under 20 characters")

df = df[msg_len >= 20].reset_index(drop=True)
print("Shape after dropping near-empty messages:", df.shape)

Dropping 246 rows with message body under 20 characters
Shape after dropping near-empty messages: (13692, 9)


## 8. Normalize the `To` field (audit #7)

In [10]:
KNOWN_LIST_MAP = {
    "tovima": "tovima", "tovima university": "tovima", "tovima@upatras.gr": "tovima",
    "tovima_upatras": "tovima", "to vima": "tovima", "το βημα": "tovima", "το βήμα": "tovima",
    "πανεπιστημίου το βήμα": "tovima", "tovima upatras": "tovima",
    "announcements": "announcements", "ανακοινωσεις": "announcements", "ανακοινώσεις": "announcements",
    "upatras announcements": "announcements", "announces": "announcements", "anakoinwseis30": "announcements",
    "studentnews": "studentnews", "μειλ φοιτητων": "studentnews",
    "sdp": "sdp",
    "all": "all",
    "info": "info",
}


def clean_to_fragment(raw):
    """Cut off digest/topic leakage and stray quote characters."""
    s = str(raw).strip()
    s = re.split(r"(Old-Topics:|New-Topics:|CC:)", s)[0]
    return s.strip().strip("'\"").strip()


def split_recipients(s):
    parts = [p.strip().strip("'\"").strip() for p in s.split(",")]
    return [p for p in parts if p]


def canonicalize_recipients(raw):
    cleaned = clean_to_fragment(raw)
    members = split_recipients(cleaned)
    canon_lists = set()
    other = []
    for m in members:
        key = m.lower()
        if key in KNOWN_LIST_MAP:
            canon_lists.add(KNOWN_LIST_MAP[key])
        else:
            other.append(m)
    return sorted(canon_lists), other


parsed = df["To"].apply(canonicalize_recipients)
df["to_lists"] = parsed.apply(lambda x: x[0])
df["to_other_recipients"] = parsed.apply(lambda x: x[1])

In [11]:
from collections import Counter

list_counts = Counter()
for lists in df["to_lists"]:
    for l in lists:
        list_counts[l] += 1

print("=== Canonical list membership counts ===")
for name, cnt in list_counts.most_common():
    print(f"{name:>15}: {cnt} ({cnt/len(df)*100:.1f}%)")

n_no_known_list = (df["to_lists"].apply(len) == 0).sum()
print(f"\nRows matching no known list (only individual/department CCs): {n_no_known_list} ({n_no_known_list/len(df)*100:.1f}%)")

=== Canonical list membership counts ===
         tovima: 9450 (69.0%)
  announcements: 4990 (36.4%)
    studentnews: 544 (4.0%)
            sdp: 287 (2.1%)
           info: 210 (1.5%)
            all: 118 (0.9%)

Rows matching no known list (only individual/department CCs): 432 (3.2%)


`to_lists`: multi-label list column. `to_other_recipients`: individual-name tail.

## 9. Subject tag [TOVIMA]/[ANNOUNCEMENTS] cross-check

In [12]:
df["subject_tag"] = df["Subject"].str.extract(r"^\[([A-Za-zΑ-Ωα-ω]+)\]")[0].str.lower()
print(df["subject_tag"].value_counts(dropna=False))

subject_tag
tovima           9674
announcements    3974
NaN                44
Name: count, dtype: int64


## 10. PII flag for embedded contact info (audit #8)

In [13]:
email_pattern = re.compile(r"[\w\.-]+@[\w\.-]+\.\w+")
phone_pattern = re.compile(r"\b(69\d{8}|2\d{9})\b")

df["contains_pii_pattern"] = (
    df["Message"].fillna("").str.contains(email_pattern, regex=True)
    | df["Message"].fillna("").str.contains(phone_pattern, regex=True)
)
print(f"Rows flagged for embedded contact info: {df['contains_pii_pattern'].sum()} ({df['contains_pii_pattern'].mean()*100:.1f}%)")

C:\Users\Jim\AppData\Local\Temp\ipykernel_13700\1436804015.py:6: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  | df["Message"].fillna("").str.contains(phone_pattern, regex=True)


Rows flagged for embedded contact info: 1380 (10.1%)


## 11. Final structural summary

In [14]:
n_end = len(df)
print(f"Rows at start:                {n_start}")
print(f"Rows after cleaning:          {n_end}")
print(f"Total dropped:                {n_start - n_end} ({(n_start - n_end)/n_start*100:.1f}%)")
print()
print("Breakdown of drops:")
print(f"  unparseable dates:          {n_bad_dates}")
print(f"  2017 partial year:          {n_2017}")
print(f"  content duplicates:         {n_before - n_after}")
print(f"  near-empty messages:        {n_short}")
print()
print("Flag columns added (no rows dropped for these):")
print("  had_html_markup, is_reply_or_forward, to_lists, to_other_recipients,")
print("  subject_tag, contains_pii_pattern")
print()
print("Final columns:", df.columns.tolist())
df.head(3)

Rows at start:                14277
Rows after cleaning:          13692
Total dropped:                585 (4.1%)

Breakdown of drops:
  unparseable dates:          3
  2017 partial year:          10
  content duplicates:         326
  near-empty messages:        246

Flag columns added (no rows dropped for these):
  had_html_markup, is_reply_or_forward, to_lists, to_other_recipients,
  subject_tag, contains_pii_pattern

Final columns: ['Author', 'Date', 'To', 'Subject', 'Message', 'Date_parsed', 'Year', 'had_html_markup', 'is_reply_or_forward', 'to_lists', 'to_other_recipients', 'subject_tag', 'contains_pii_pattern']


,Author,Date,To,Subject,Message,Date_parsed,Year,had_html_markup,is_reply_or_forward,to_lists,to_other_recipients,subject_tag,contains_pii_pattern
0,ΠΑΝΕΠΙΣΤΗΜΙΑΚΗ ΕΝΗΜΕΡΩΣΗ,2018-01-03 10:10 UTC,announcements,[ANNOUNCEMENTS] Ανακοίνωση για θέση PhD στην Ελβετία,Επισυνάπτεται ανακοίνωση για θέση PhD στην Ελβετία,2018-01-03 10:10:00+00:00,2018,False,False,[announcements],[],announcements,False
1,Dr. Dimitris Zois,2018-01-04 09:34 UTC,tovima,[TOVIMA] Συνταξιοδότηση Μηχανικών - μελών ΔΕΠ,Ενημερωτικό :\n\nΣτις*29-12-2017* εκδόθηκε από το Υπουργείο Εργασίας η πολυα...,2018-01-04 09:34:00+00:00,2018,False,False,[tovima],[],tovima,False
2,Πανεπιστημιακός Ι.Ναός,2018-01-04 17:51 UTC,TovimaOld-Topics: [TOVIMA] Πρόγραμμα ἰερῶν ἀκολουθιῶν Δεκεμβρίου 2017N...,[TOVIMA] Πρόγραμμα ἰερῶν ἀκολουθιῶν Ἰανουαρίου 2018,Ἐπισυνάπτεται ὠς ἀρχεῖο εἰκόνας. \n\nhttp://inaos.upatras.gr/ [1] \n\n\n\nLi...,2018-01-04 17:51:00+00:00,2018,False,False,[tovima],[],tovima,False


## 12. Save

UTF-32, tab-separated — same format as the rest of the pipeline.

In [15]:
# list columns stringified for CSV; parse back with json.loads
import json

df_out = df.copy()
df_out["to_lists"] = df_out["to_lists"].apply(json.dumps)
df_out["to_other_recipients"] = df_out["to_other_recipients"].apply(json.dumps)

df_out.to_csv("tovima_clean.csv", sep="\t", encoding="utf-32", index=False)
print("Saved tovima_clean.csv:", df_out.shape)

Saved tovima_clean.csv: (13692, 13)
